In [1]:
# Get the library files
import pandas as pd
import numpy as np

In [6]:
# Load the dataset
dataset = pd.read_csv("DillibabuSarva_DefectDataset.csv")

In [7]:
# Display the dataset
dataset

,SHA,cbo,wmc,dit,rfc,lcom,totalMethods,totalFields,nosi,loc,...,tryCatchQty,parenthesizedExpsQty,stringLiteralsQty,numbersQty,assignmentsQty,mathOperationsQty,variablesQty,maxNestedBlocks,uniqueWordsQty,defect
0,7a955fd6c7de2bd912be544dcfe77f9173a7aa600,5,60,2,55,189,27,5,30,247,...,4,2,47,9,27,5,17,3,191,0
1,000f1ab4780fc9460975791c52597f7c04e15be70,3,10,1,1,9,7,4,1,38,...,0,0,0,22,4,0,4,2,69,0
2,000f1ab4780fc9460975791c52597f7c04e15be71,3,10,1,1,9,7,4,0,38,...,0,0,0,22,4,0,4,2,69,1
3,0024dbdd6ba3cc7797cc0b1ae537dcdc488c4c270,20,59,3,63,189,24,9,4,262,...,0,6,6,14,45,8,41,4,222,0
4,0024dbdd6ba3cc7797cc0b1ae537dcdc488c4c271,21,58,2,61,189,24,9,0,260,...,0,6,6,14,45,8,41,4,222,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6047,ffd1ed788cbf10bed00d49d79c7ee44250c36ac11,52,124,12,144,963,110,9,0,804,...,0,0,26,16,32,4,30,6,689,1
6048,ffdf4a3fcccb7489548c6a2ff6af7cebc92a180c0,24,27,2,46,4,8,8,0,126,...,0,1,3,14,27,0,24,3,108,0
6049,ffdf4a3fcccb7489548c6a2ff6af7cebc92a180c1,22,27,1,46,4,8,8,0,126,...,0,1,3,14,27,0,24,3,108,1
6050,ffe7c9989a4553d35fd1d5041d0cece0a673a0c80,3,12,2,12,28,8,0,1,67,...,2,0,0,2,10,0,8,2,36,0


In [8]:
# Display the column names
dataset.columns

Index(['SHA', 'cbo', 'wmc', 'dit', 'rfc', 'lcom', 'totalMethods',
       'totalFields', 'nosi', 'loc', 'returnQty', 'loopQty', 'comparisonsQty',
       'tryCatchQty', 'parenthesizedExpsQty', 'stringLiteralsQty',
       'numbersQty', 'assignmentsQty', 'mathOperationsQty', 'variablesQty',
       'maxNestedBlocks', 'uniqueWordsQty', 'defect'],
      dtype='object')

In [9]:
# Feature selection result
selected_features = ['nosi','dit','cbo','rfc','maxNestedBlocks',
                     'uniqueWordsQty','assignmentsQty','numbersQty',
                     'tryCatchQty','parenthesizedExpsQty']

X = dataset[selected_features]
y = dataset['defect']

In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

In [13]:
from sklearn.ensemble import AdaBoostClassifier
classifier = AdaBoostClassifier(estimator=None, n_estimators=50, learning_rate=1.0, algorithm='SAMME', random_state=None)
#fitting the model for grid search
classifier.fit(X_train, y_train)

AdaBoostClassifier(algorithm='SAMME')

In [14]:
y_pred = classifier.predict(X_test)

In [15]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)

In [16]:
print(cm)

[[590 318]
 [118 790]]


In [17]:
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, y_pred)

In [18]:
# AdaBoostClassification Report
print(clf_report)

              precision    recall  f1-score   support

           0       0.83      0.65      0.73       908
           1       0.71      0.87      0.78       908

    accuracy                           0.76      1816
   macro avg       0.77      0.76      0.76      1816
weighted avg       0.77      0.76      0.76      1816



In [23]:
# Finding the outliers for our dataset
Q1 = dataset[selected_features].quantile(0.25)
Q3 = dataset[selected_features].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR
#Check
for col in selected_features:
    less = (dataset[col] < lower[col]).sum()
    great = (dataset[col] > upper[col]).sum()
    print(f"{col}: Lesser = {less}, Greater = {great}")

nosi: Lesser = 0, Greater = 967
dit: Lesser = 0, Greater = 788
cbo: Lesser = 0, Greater = 403
rfc: Lesser = 0, Greater = 367
maxNestedBlocks: Lesser = 0, Greater = 411
uniqueWordsQty: Lesser = 0, Greater = 431
assignmentsQty: Lesser = 0, Greater = 490
numbersQty: Lesser = 0, Greater = 651
tryCatchQty: Lesser = 0, Greater = 651
parenthesizedExpsQty: Lesser = 0, Greater = 671


In [24]:
# Replacing the outliers with mean values
for col in selected_features:
    mean = dataset[col].mean()
    dataset[col] = np.where((dataset[col] > upper[col]) | (dataset[col] < lower[col]), mean, dataset[col])

In [25]:
# After replacing the outliers
for col in selected_features:
    less = (dataset[col] < lower[col]).sum()
    great = (dataset[col] > upper[col]).sum()
    print(f"{col}: Lesser = {less}, Greater = {great}")

nosi: Lesser = 0, Greater = 0
dit: Lesser = 0, Greater = 0
cbo: Lesser = 0, Greater = 0
rfc: Lesser = 0, Greater = 0
maxNestedBlocks: Lesser = 0, Greater = 0
uniqueWordsQty: Lesser = 0, Greater = 0
assignmentsQty: Lesser = 0, Greater = 0
numbersQty: Lesser = 0, Greater = 0
tryCatchQty: Lesser = 0, Greater = 0
parenthesizedExpsQty: Lesser = 0, Greater = 0


In [26]:
A = dataset[selected_features]
b = dataset['defect']
A_train, A_test, b_train, b_test = train_test_split(A, b, test_size = 0.3, random_state = 42, stratify = b)

In [28]:
classifier_recheck = AdaBoostClassifier(estimator = None, n_estimators = 50, learning_rate = 1.0, algorithm = "SAMME", random_state = None)
# fitting the model for grid search
classifier_recheck.fit(A_train, b_train)

AdaBoostClassifier(algorithm='SAMME')

In [30]:
b_pred = classifier_recheck.predict(A_test)

In [31]:
cmodel = confusion_matrix(b_test,b_pred)
print(cmodel)

[[610 298]
 [123 785]]


In [32]:
# AdaBoostClassification Report after replacing outliers
clf_report_check = classification_report(b_test, b_pred)
print(clf_report_check)

              precision    recall  f1-score   support

           0       0.83      0.67      0.74       908
           1       0.72      0.86      0.79       908

    accuracy                           0.77      1816
   macro avg       0.78      0.77      0.77      1816
weighted avg       0.78      0.77      0.77      1816



In [33]:
# Here, we could see accuracy increased by 1%

In [38]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier

In [39]:
# 1. Create a synthetic dataset
A, b = make_classification(n_samples=1000, n_features=20, random_state=42)
A_train, A_test, b_train, b_test = train_test_split(A, b, test_size=0.2, random_state=42)

In [47]:
# 2. Instantiate the base estimator and the AdaBoost model
# Note: 'estimator' replaced the deprecated 'base_estimator' parameter in scikit-learn
base_tree = DecisionTreeClassifier(max_depth=1) 
ada = AdaBoostClassifier(estimator=base_tree, algorithm = 'SAMME', random_state=42)

In [48]:
# 3. Define the parameter grid to explore
param_grid = {
    'n_estimators': [50, 100, 200, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0],
    'algorithm': ['SAMME'],
    'estimator__max_depth': [1, 2]  # Accesses the base estimator's parameters directly
}

In [49]:
# 4. Set up the Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=ada, 
    param_grid=param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1, 
    verbose=1
)

In [50]:
# 5. Fit the grid search to the training data
grid_search.fit(A_train, b_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


GridSearchCV(cv=5,
             estimator=AdaBoostClassifier(algorithm='SAMME',
                                          estimator=DecisionTreeClassifier(max_depth=1),
                                          random_state=42),
             n_jobs=-1,
             param_grid={'algorithm': ['SAMME'], 'estimator__max_depth': [1, 2],
                         'learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0],
                         'n_estimators': [50, 100, 200, 500]},
             scoring='accuracy', verbose=1)

In [51]:
# 1. Convert the cv_results_ dictionary into a pandas DataFrame
results_df = pd.DataFrame(grid_search.cv_results_)

In [53]:
# 2. Filter and sort to see the best combinations at the top
important_columns = [
    'param_n_estimators', 
    'param_learning_rate', 
    'param_estimator__max_depth', 
    'mean_test_score', 
    'std_test_score', 
    'mean_fit_time'
]

# View the top 5 performing parameter combinations
display(results_df[important_columns].sort_values(by='mean_test_score', ascending=False))

,param_n_estimators,param_learning_rate,param_estimator__max_depth,mean_test_score,std_test_score,mean_fit_time
30,200,0.10,2,0.90000,0.028777,0.904201
27,500,0.05,2,0.90000,0.031125,2.417120
31,500,0.10,2,0.89375,0.027951,2.305527
34,200,0.50,2,0.89375,0.027099,0.920282
33,100,0.50,2,0.89125,0.026101,0.438914
35,500,0.50,2,0.88625,0.032452,2.352433
32,50,0.50,2,0.88500,0.019605,0.219010
39,500,1.00,2,0.88500,0.021506,2.182336
38,200,1.00,2,0.88500,0.022569,0.930724
29,100,0.10,2,0.88375,0.023251,0.466875


In [54]:
# 6. Extract and evaluate results
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Score: {grid_search.best_score_:.4f}")

Best Hyperparameters: {'algorithm': 'SAMME', 'estimator__max_depth': 2, 'learning_rate': 0.05, 'n_estimators': 500}
Best Cross-Validation Score: 0.9000


In [56]:
# Evaluate performance on the untouched test set
best_model = grid_search.best_estimator_
test_accuracy = best_model.score(A_test, b_test)
print(f"Test Set Accuracy: {test_accuracy:.4f}")

Test Set Accuracy: 0.8750


In [57]:
#importing pickle library for deployment phase
import pickle

In [58]:
#here we are assigning the saving model file name with extension to the filename variable
filename = "finalized-model_AdaBoost_Classification_Defect_Prediction.sav"
#using pickle dump method we are writing the file on disk
pickle.dump(classifier, open(filename, 'wb'))

In [64]:
import warnings
warnings.filterwarnings("ignore")
with open("finalized-model_AdaBoost_Classification_Defect_Prediction.sav", "rb") as f:
    model = pickle.load(f)
new_prediction = model.predict([[10, 5, 20, 70, 5, 200, 60, 30, 3, 10]])
print(f"Defect? {new_prediction[0]}")

Defect? 0


In [65]:
# Here after removing outliers, we could see accuracy we are getting as 87.5%